# Fall Classification V2 - K-Fold Ensemble with Optimized Training

**V2 Strategy (Best Approach for Fall Detection):**
- 5-fold video-level cross-validation
- Partial freeze (only blocks 5-6 + head)
- Strong regularization (dropout=0.5, drop_path=0.2, weight_decay=0.05, label_smoothing=0.1)
- MixUp augmentation (alpha=0.4)
- Square-root class balancing for better precision/recall balance
- Test-Time Augmentation (horizontal flip)

**Expected:** Val F1 0.70-0.78 per fold, Ensemble Test F1 0.78-0.84

## Cell 1: Setup

In [ ]:
!pip install -q timm torch torchvision scikit-learn matplotlib seaborn

import os
import sys
import io
import json
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.model_selection import KFold

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Paths
DATA_ROOT = Path('/kaggle/input/fall-classification')
OUTPUT_ROOT = Path('/kaggle/working')
OUTPUT_ROOT.mkdir(exist_ok=True)

# Suppress DataLoader worker shutdown exception (keep num_workers=2 for speed)
class _FilterDataLoaderShutdown:
    def __init__(self, stream):
        self.stream = stream
        self._skip = False
    def write(self, msg):
        if "_MultiProcessingDataLoaderIter" in msg:
            self._skip = True
        if self._skip:
            if "AssertionError" in msg and "child process" in msg:
                self._skip = False
            return
        self.stream.write(msg)
    def flush(self):
        self.stream.flush()
sys.stderr = _FilterDataLoaderShutdown(sys.stderr)

## Cell 2: Load Dataset & Extract Video IDs

In [ ]:
# Load train+val (we'll split them ourselves for k-fold)
# Test set is already held out and will NOT be used in k-fold
full_dataset = ImageFolder(DATA_ROOT / 'train')
test_dataset = ImageFolder(DATA_ROOT / 'test')

print(f"Full trainval samples: {len(full_dataset)}")
print(f"Test samples (held out): {len(test_dataset)}")
print(f"Classes: {full_dataset.classes}")  # ['fall', 'normal']

# Extract video IDs from filenames
# Filename format: fall_video123_frame045.jpg -> video ID = fall_video123
import re

def extract_video_id(filepath):
    """Extract video ID from triplet filename"""
    filename = Path(filepath).stem  # Remove extension
    # Remove frame suffix (e.g., _triplet_0045 -> video ID)
    match = re.match(r'^(.+)_triplet_\d+$', filename)
    if match:
        return match.group(1)
    # Fallback: just use the filename up to last underscore
    parts = filename.rsplit('_', 1)
    return parts[0] if len(parts) > 1 else filename

# Group samples by video ID
video_to_indices = defaultdict(list)
for idx, (filepath, label) in enumerate(full_dataset.samples):
    video_id = extract_video_id(filepath)
    video_to_indices[video_id].append(idx)

videos = sorted(video_to_indices.keys())
print(f"\nUnique videos in trainval: {len(videos)}")
print(f"Example videos: {videos[:5]}")

# Verify video distribution
for video in videos[:5]:
    indices = video_to_indices[video]
    print(f"  {video}: {len(indices)} triplets")

## Cell 3: Data Transforms with MixUp

In [ ]:
# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# MixUp augmentation
class MixUpAugmentation:
    """MixUp: blend pairs of images and labels"""
    def __init__(self, alpha=0.4):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels):
        """Apply MixUp to a batch"""
        if self.alpha <= 0:
            return batch_images, batch_labels
        
        batch_size = batch_images.size(0)
        lam = np.random.beta(self.alpha, self.alpha)
        
        # Random permutation
        index = torch.randperm(batch_size).to(batch_images.device)
        
        # Mix images
        mixed_images = lam * batch_images + (1 - lam) * batch_images[index]
        
        # Mix labels (soft labels)
        labels_a = batch_labels
        labels_b = batch_labels[index]
        
        return mixed_images, labels_a, labels_b, lam

mixup = MixUpAugmentation(alpha=0.4)

# Training transforms
# CRITICAL: saturation and hue MUST be 0 because color channels encode time!
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.0,  # MUST be 0
        hue=0.0          # MUST be 0
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation/test transforms
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# TTA transform (horizontal flip)
tta_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=1.0),  # Always flip
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Transforms and MixUp defined")

## Cell 4: K-Fold Split (Video-Level)

In [ ]:
# 5-fold CV at video level
N_FOLDS = 5
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Create fold splits
fold_splits = []
for fold_idx, (train_videos_idx, val_videos_idx) in enumerate(kfold.split(videos)):
    train_videos = [videos[i] for i in train_videos_idx]
    val_videos = [videos[i] for i in val_videos_idx]
    
    # Get sample indices for these videos
    train_indices = []
    val_indices = []
    
    for video in train_videos:
        train_indices.extend(video_to_indices[video])
    
    for video in val_videos:
        val_indices.extend(video_to_indices[video])
    
    fold_splits.append({
        'fold': fold_idx,
        'train_videos': train_videos,
        'val_videos': val_videos,
        'train_indices': train_indices,
        'val_indices': val_indices
    })
    
    print(f"Fold {fold_idx}: Train={len(train_videos)} videos ({len(train_indices)} samples), "
          f"Val={len(val_videos)} videos ({len(val_indices)} samples)")

print(f"\nK-Fold splits created")

## Cell 5: Training Configuration

In [ ]:
# Training config
BATCH_SIZE = 32
NUM_WORKERS = 2  # stderr filter in Cell 1 suppresses shutdown message
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 30
EARLY_STOP_PATIENCE = 12
GRAD_CLIP = 1.0

print(f"Config: {N_FOLDS} folds, Phase1={PHASE1_EPOCHS}ep, Phase2={PHASE2_EPOCHS}ep")
print(f"Batch size: {BATCH_SIZE}, MixUp alpha: 0.4")
print(f"Regularization: dropout=0.5, drop_path=0.2, weight_decay=0.05, label_smoothing=0.1")

## Cell 6: Training Functions with MixUp

In [ ]:
def train_one_epoch_mixup(model, loader, optimizer, criterion, device, use_mixup=True):
    """Train for one epoch with MixUp"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Apply MixUp
        if use_mixup and random.random() < 0.8:  # 80% of batches use MixUp
            mixed_images, labels_a, labels_b, lam = mixup(images, labels)
            
            optimizer.zero_grad()
            outputs = model(mixed_images)
            # MixUp loss: weighted combination
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            # Approximate accuracy for MixUp
            correct += (lam * predicted.eq(labels_a).sum().item() + 
                       (1-lam) * predicted.eq(labels_b).sum().item())
        else:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': running_loss / (pbar.n + 1),
            'acc': 100.0 * correct / total
        })
    
    return running_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion, device, class_to_idx):
    """Evaluate model"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    fall_idx = class_to_idx['fall']
    
    return {
        'loss': running_loss / len(loader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='binary', pos_label=fall_idx, zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='binary', pos_label=fall_idx, zero_division=0),
        'f1': f1_score(all_labels, all_preds, average='binary', pos_label=fall_idx, zero_division=0),
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs
    }

print("Training functions defined")

## Cell 7: Train All Folds

In [ ]:
fold_results = []

for fold_data in fold_splits:
    fold_idx = fold_data['fold']
    print("\n" + "=" * 80)
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print("=" * 80)
    print(f"Train: {len(fold_data['train_videos'])} videos, {len(fold_data['train_indices'])} samples")
    print(f"Val: {len(fold_data['val_videos'])} videos, {len(fold_data['val_indices'])} samples")
    
    # Create datasets for this fold
    full_dataset.transform = train_transform
    train_subset = Subset(full_dataset, fold_data['train_indices'])
    
    full_dataset.transform = eval_transform
    val_subset = Subset(full_dataset, fold_data['val_indices'])
    
    # Square-root class balancing for WeightedRandomSampler
    train_labels = [full_dataset.samples[i][1] for i in fold_data['train_indices']]
    class_counts = [train_labels.count(i) for i in range(len(full_dataset.classes))]
    n_total = sum(class_counts)
    
    # Square root balancing (less aggressive than inverse frequency)
    class_weights_sampler = [np.sqrt(n_total / count) if count > 0 else 0 for count in class_counts]
    sample_weights = [class_weights_sampler[label] for label in train_labels]
    
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    
    # Square-root class weights for loss
    loss_weights = torch.tensor(
        [np.sqrt(n_total / count) if count > 0 else 0 for count in class_counts],
        dtype=torch.float32
    ).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=0.1)
    
    print(f"Class counts: fall={class_counts[0]}, normal={class_counts[1]}")
    print(f"Sqrt loss weights: {loss_weights.tolist()}")
    
    # DataLoaders
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler,
                             num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
    
    # Create model with strong regularization
    model = timm.create_model(
        'efficientnet_b0',
        pretrained=True,
        num_classes=2,
        drop_rate=0.5,
        drop_path_rate=0.2
    ).to(device)
    
    # PHASE 1: Head only
    print("\n--- Phase 1: Train head only ---")
    for name, param in model.named_parameters():
        param.requires_grad = 'classifier' in name
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,
        weight_decay=0.05
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)
    
    best_val_f1 = 0.0
    fold_history = {'train_loss': [], 'val_f1': []}
    
    for epoch in range(PHASE1_EPOCHS):
        train_loss, train_acc = train_one_epoch_mixup(model, train_loader, optimizer, criterion, device, use_mixup=True)
        val_metrics = evaluate(model, val_loader, criterion, device, full_dataset.class_to_idx)
        scheduler.step()
        
        fold_history['train_loss'].append(train_loss)
        fold_history['val_f1'].append(val_metrics['f1'])
        
        print(f"Epoch {epoch+1}/{PHASE1_EPOCHS}: Loss={train_loss:.3f}, TrainAcc={train_acc:.1f}%, "
              f"ValF1={val_metrics['f1']:.4f}, ValAcc={val_metrics['accuracy']*100:.1f}%")
        
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), OUTPUT_ROOT / f'best_fold{fold_idx}_phase1.pt')
    
    # PHASE 2: Partial unfreeze (only blocks 5-6 + head)
    print("\n--- Phase 2: Unfreeze blocks 5-6 + head ---")
    model.load_state_dict(torch.load(OUTPUT_ROOT / f'best_fold{fold_idx}_phase1.pt'))
    
    # Unfreeze only last 2 blocks (5, 6) + conv_head + bn2 + classifier
    for name, param in model.named_parameters():
        if any(key in name for key in ['blocks.5.', 'blocks.6.', 'conv_head', 'bn2', 'classifier']):
            param.requires_grad = True
        else:
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,}")
    
    # Differential LR
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' not in n]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' in n]
    
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': 5e-6},
        {'params': head_params, 'lr': 5e-5}
    ], weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)
    
    patience_counter = 0
    
    for epoch in range(PHASE2_EPOCHS):
        train_loss, train_acc = train_one_epoch_mixup(model, train_loader, optimizer, criterion, device, use_mixup=True)
        val_metrics = evaluate(model, val_loader, criterion, device, full_dataset.class_to_idx)
        scheduler.step()
        
        fold_history['train_loss'].append(train_loss)
        fold_history['val_f1'].append(val_metrics['f1'])
        
        print(f"Epoch {epoch+1}/{PHASE2_EPOCHS}: Loss={train_loss:.3f}, TrainAcc={train_acc:.1f}%, "
              f"ValF1={val_metrics['f1']:.4f}, ValAcc={val_metrics['accuracy']*100:.1f}%, "
              f"ValRecall={val_metrics['recall']:.4f}, ValPrec={val_metrics['precision']:.4f}")
        
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), OUTPUT_ROOT / f'best_fold{fold_idx}.pt')
            print(f"  ✓ Saved (F1={best_val_f1:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stop at epoch {epoch+1}")
                break
    
    fold_results.append({
        'fold': fold_idx,
        'best_val_f1': best_val_f1,
        'val_videos': fold_data['val_videos'],
        'history': fold_history
    })
    
    print(f"\nFold {fold_idx} complete. Best Val F1: {best_val_f1:.4f}")

print("\n" + "=" * 80)
print("ALL FOLDS COMPLETE")
print("=" * 80)
for r in fold_results:
    print(f"Fold {r['fold']}: Val F1 = {r['best_val_f1']:.4f}")
print(f"\nMean Val F1 across folds: {np.mean([r['best_val_f1'] for r in fold_results]):.4f}")

## Cell 8: Ensemble Evaluation on Test Set with TTA

In [ ]:
print("\n" + "=" * 80)
print("ENSEMBLE EVALUATION ON TEST SET WITH TTA")
print("=" * 80)

# Load all 5 fold models
models = []
for fold_idx in range(N_FOLDS):
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2,
                             drop_rate=0.5, drop_path_rate=0.2)
    model.load_state_dict(torch.load(OUTPUT_ROOT / f'best_fold{fold_idx}.pt'))
    model.to(device)
    model.eval()
    models.append(model)
    print(f"✓ Loaded fold {fold_idx} model")

print(f"\nTest set: {len(test_dataset)} samples")
print("Running ensemble inference with TTA (horizontal flip)...\n")

# Ensemble prediction with TTA
all_labels = []
all_ensemble_probs = []

# Create test loader without transform (we'll apply manually for TTA)
from PIL import Image

with torch.no_grad():
    for idx in tqdm(range(len(test_dataset)), desc='Ensemble Test + TTA'):
        img_path, label = test_dataset.samples[idx]
        img = Image.open(img_path).convert('RGB')
        
        # Original + flipped
        img_orig = eval_transform(img).unsqueeze(0).to(device)
        img_flip = tta_transform(img).unsqueeze(0).to(device)
        
        # Get predictions from all 5 models on both versions
        fold_probs = []
        for model in models:
            # Original
            outputs_orig = model(img_orig)
            probs_orig = torch.softmax(outputs_orig, dim=1)
            
            # Flipped
            outputs_flip = model(img_flip)
            probs_flip = torch.softmax(outputs_flip, dim=1)
            
            # Average original and flipped
            fold_probs.append((probs_orig + probs_flip) / 2.0)
        
        # Average across all 5 models
        ensemble_probs = torch.stack(fold_probs).mean(dim=0)
        
        all_labels.append(label)
        all_ensemble_probs.append(ensemble_probs.cpu().numpy()[0])

all_ensemble_probs = np.array(all_ensemble_probs)

# Compute ensemble metrics
ensemble_preds = np.argmax(all_ensemble_probs, axis=1)
fall_idx = test_dataset.class_to_idx['fall']

test_acc = accuracy_score(all_labels, ensemble_preds)
test_precision = precision_score(all_labels, ensemble_preds, pos_label=fall_idx, zero_division=0)
test_recall = recall_score(all_labels, ensemble_preds, pos_label=fall_idx, zero_division=0)
test_f1 = f1_score(all_labels, ensemble_preds, pos_label=fall_idx, zero_division=0)

print("\n" + "=" * 80)
print("ENSEMBLE TEST RESULTS (with TTA)")
print("=" * 80)
print(f"Accuracy:  {test_acc*100:.2f}%")
print(f"Precision: {test_precision:.4f}")
print(f"Recall (Fall): {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")

# Confusion matrix
cm = confusion_matrix(all_labels, ensemble_preds)
print(f"\nConfusion Matrix:")
print(cm)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=test_dataset.classes, yticklabels=test_dataset.classes)
plt.title(f'Ensemble Test Confusion Matrix (F1={test_f1:.3f})')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.savefig(OUTPUT_ROOT / 'ensemble_confusion_matrix.png', dpi=150)
plt.show()

# Save metrics
metrics = {
    'ensemble_test': {
        'accuracy': float(test_acc),
        'precision': float(test_precision),
        'recall': float(test_recall),
        'f1': float(test_f1),
        'used_tta': True
    },
    'fold_results': [
        {'fold': r['fold'], 'val_f1': r['best_val_f1'], 'val_videos': r['val_videos'][:10]}  # Save first 10 videos
        for r in fold_results
    ],
    'mean_val_f1': float(np.mean([r['best_val_f1'] for r in fold_results]))
}

with open(OUTPUT_ROOT / 'ensemble_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n✓ Metrics saved")

## Cell 9: Export All Models

In [ ]:
export_dir = OUTPUT_ROOT / 'fall_v2_ensemble'
export_dir.mkdir(exist_ok=True)

# Copy all fold models
for fold_idx in range(N_FOLDS):
    shutil.copy(OUTPUT_ROOT / f'best_fold{fold_idx}.pt', export_dir / f'fold{fold_idx}.pt')

# Copy metrics and plots
shutil.copy(OUTPUT_ROOT / 'ensemble_metrics.json', export_dir / 'metrics.json')
shutil.copy(OUTPUT_ROOT / 'ensemble_confusion_matrix.png', export_dir / 'confusion_matrix.png')

# Create README
readme = f"""# Fall Classifier V2 - Ensemble

**Architecture:** EfficientNet-B0 (5 models ensemble)
**Encoding:** Temporal RGB triplets (t-1, t, t+1 frames)
**Training:** 5-fold video-level CV with MixUp, partial freeze (blocks 5-6 only), strong regularization
**Class Balancing:** Square-root weighting for better precision/recall balance
**TTA:** Horizontal flip at test time

## Results

- Mean Val F1: {metrics['mean_val_f1']:.4f}
- Ensemble Test F1: {metrics['ensemble_test']['f1']:.4f}
- Test Accuracy: {metrics['ensemble_test']['accuracy']*100:.2f}%
- Test Recall (Fall): {metrics['ensemble_test']['recall']:.4f}
- Test Precision: {metrics['ensemble_test']['precision']:.4f}

## Usage

```python
# Load all 5 models
models = []
for i in range(5):
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2,
                             drop_rate=0.5, drop_path_rate=0.2)
    model.load_state_dict(torch.load(f'fold{{i}}.pt'))
    model.eval()
    models.append(model)

# Ensemble inference
with torch.no_grad():
    fold_probs = [torch.softmax(m(image), dim=1) for m in models]
    ensemble_prob = torch.stack(fold_probs).mean(dim=0)
```

## Deployment

Extract all 5 `fold*.pt` files to:
```
d:/project/FYP/fall_detection/weights/
  fold0.pt
  fold1.pt
  fold2.pt
  fold3.pt
  fold4.pt
```
"""

with open(export_dir / 'README.md', 'w') as f:
    f.write(readme)

# Zip for download
shutil.make_archive(str(OUTPUT_ROOT / 'fall_v2_ensemble'), 'zip', export_dir)

print(f"\n✓ Exported to fall_v2_ensemble.zip")
print(f"\nDownload and extract 5 fold models to:")
print(f"  d:/project/FYP/fall_detection/weights/fold{{0-4}}.pt")